In [1]:
import ete3
from glob import glob
from collections import defaultdict
from ete3 import PhyloTree
from ete3 import Tree

In [2]:
chronos={}
#chronoGenes={}
for c in glob("*_sample.chronogram"):
    og="OG_"+c.split('_')[1]
    ts=open(c).readline()
    print(og)
    g2s=defaultdict(list)
    t=PhyloTree(ts,format=1,sp_naming_function=lambda x:x.split("_")[0])
    for l in t:
        sp,gid=l.name.split('_',1)
        gid=gid.split('.')[0]
        l.add_features(species=sp,gid=gid)
        g2s[sp].append(gid)
    #chronoGenes[og]=g2s
    for nd in t.traverse("postorder"):
        if not nd.is_leaf():
            oneChild=nd.get_closest_leaf(topology_only=False, is_leaf_fn=None)
            nd.add_features(age=oneChild[1])
    chronos[og]=t


OG_20070
OG_4516
OG_251
OG_15801
OG_8669
OG_2551
OG_150
OG_7386
OG_18117
OG_7389
OG_14227
OG_13015
OG_18015
OG_10478
OG_6682
OG_19899
OG_6946
OG_6651
OG_10010
OG_1920
OG_3635
OG_15304
OG_8512
OG_3837
OG_10714
OG_6871
OG_4926
OG_4216
OG_9610
OG_6633
OG_11275
OG_5680
OG_9680
OG_1101
OG_3943
OG_4603
OG_1557
OG_5020
OG_3056
OG_10726
OG_11351
OG_4277
OG_7428
OG_16364
OG_19347
OG_19962
OG_20153
OG_6010
OG_6776
OG_10660
OG_7427
OG_2773
OG_11054
OG_4608
OG_5024
OG_3147
OG_8647
OG_1351
OG_6836
OG_6981
OG_7973
OG_9626
OG_8961
OG_10875
OG_1736
OG_19944
OG_6876
OG_10771
OG_5162
OG_1968
OG_1128
OG_846
OG_1032
OG_3429
OG_11280
OG_48
OG_806
OG_10942
OG_11081
OG_6830
OG_813
OG_4032
OG_642
OG_1595
OG_4438
OG_11114
OG_10749
OG_225
OG_2102
OG_503
OG_10205
OG_9287
OG_341
OG_7556
OG_9587
OG_1063
OG_3945
OG_2766
OG_19880
OG_14185
OG_10980
OG_9290
OG_1498
OG_19798
OG_6161
OG_7230
OG_1209
OG_11549
OG_6654
OG_8569
OG_10742
OG_1643
OG_8529
OG_10864
OG_4703
OG_9004
OG_3843
OG_7895
OG_2821
OG_2083
OG_4514
OG_4084

In [3]:
sp_tree=PhyloTree("((((((((((Tricas,Dromel)Holometabola,Clodip)Insecta,(Dapmag,Ampamp)B)C,((Hyaazt,Penmon)Cru_A,Euraff)Crustacea)Pancrustacea,((Helhol,Tricor)Millipede,Strmar)Myriapoda)Mandibulata,((Limpol,Carrot)Xiphosura,Triant)Chelicerata)Euarthopoda,(Hypduj,Ramvar)Tardigrada)Panarthropoda,((((Caeele,Melinc)Rhbdidita,Brumal)Chromadorea,Trispi)Nem_1,Romcul)Nematoda)Ecdy_A,((Pricau,Halspi)Priapulimorpha,Tubcor)Priapulida)Ecdysozoa,((Acagra,(Pecmax,Lotgig)Conchifera)Mollusca,((Phoaur,Linana)Lophophorata,(Notgen,(Owefus,Captel)Annelida)Spiralia_A)Spiralia_B)Lophotrochozoa)Protostomia;",format=1)

In [4]:
print(sp_tree.get_ascii(show_internal=True))


                                                                                               /-Tricas
                                                                                   /Holometabola
                                                                            /Insecta           \-Dromel
                                                                           |      |
                                                                         /C|       \-Clodip
                                                                        |  |
                                                                        |  |   /-Dapmag
                                                                        |   \B|
                                                             /Pancrustacea     \-Ampamp
                                                            |           |
                                                            |           |              /-Hyaazt
               

In [5]:
recon_tree, events = t.reconcile(sp_tree)

In [6]:
with open('dat_res_clg_3.txt','w') as out: 
    for og,t in chronos.items():
        recon_tree, events = t.reconcile(sp_tree)
        for ev in events:
            if ev.etype == "D":
                #print ('   PARALOGY RELATIONSHIP:', ','.join(ev.in_seqs), "<====>", ','.join(ev.out_seqs))
                if len(ev.in_seqs)==1 or len(ev.out_seqs)==1: continue
                dn=t.get_common_ancestor(ev.in_seqs+ev.out_seqs)
                lb_in,lb_out=set([l.split('_')[-1] for l in ev.in_seqs]),set([l.split('_')[-1] for l in ev.out_seqs])
                sp_in,sp_out=set([l.split('_')[0] for l in ev.in_seqs]),set([l.split('_')[0] for l in ev.out_seqs])
                sm_in,sm_out=sp_tree.get_common_ancestor(list(sp_in)).name,sp_tree.get_common_ancestor(list(sp_out)).name
                print('d',round(dn.age,1),ev.in_seqs,ev.out_seqs,lb_in,lb_out)
                oL=[og,str(round(dn.age,1)),sm_in+'/'+sm_out,'dupl',','.join(ev.in_seqs),','.join(ev.out_seqs)]
                out.write('>>'+'\t'.join(oL)+'\n')
            if ev.etype == "S":
                #print('   ORTHOLOGY RELATIONSHIP:', ','.join(ev.in_seqs), "<====>", ','.join(ev.out_seqs))
                s0,s1=list(set([l.split('_')[0] for l in ev.in_seqs])),list(set([l.split('_')[0] for l in ev.out_seqs]))
                sm=sp_tree.get_common_ancestor([s0[0],s1[0]])
                dn=t.get_common_ancestor(ev.in_seqs+ev.out_seqs)
                out.write('\t'.join([og,str(round(dn.age,2)),sm.name,'clade',','.join(ev.in_seqs),','.join(ev.out_seqs)])+'\n')



d 515.5 ['Tubcor_g8528.t1', 'Pricau_PCA16059', 'Halspi_ScKMIYO_40.g19557'] ['Tubcor_g7176.t1', 'Pricau_PCA12894', 'Halspi_ScKMIYO_40.g19555'] {'PCA16059', '40.g19557', 'g8528.t1'} {'g7176.t1', 'PCA12894', '40.g19555'}
d 433.3 ['Limpol_XP_013773277', 'Limpol_XP_013789386', 'Carrot_SINCROG022158'] ['Limpol_XP_013787411', 'Carrot_SINCROG008582'] {'013773277', 'SINCROG022158', '013789386'} {'013787411', 'SINCROG008582'}
d 545.2 ['Tubcor_g769.t1', 'Pricau_PCA07734', 'Pricau_PCA05572', 'Halspi_ScKMIYO_412.g20778'] ['Tubcor_g10640.t1', 'Pricau_PCA33246', 'Halspi_ScKMIYO_104.g5516'] {'g769.t1', '412.g20778', 'PCA07734', 'PCA05572'} {'104.g5516', 'PCA33246', 'g10640.t1'}
d 216.5 ['Limpol_XP_013772404', 'Carrot_SINCROG022407'] ['Limpol_XP_013785017', 'Carrot_SINCROG018007'] {'SINCROG022407', '013772404'} {'SINCROG018007', '013785017'}
d 246.4 ['Limpol_XP_013772404', 'Carrot_SINCROG022407', 'Limpol_XP_013785017', 'Carrot_SINCROG018007'] ['Limpol_XP_013779670', 'Carrot_SINCROG017574'] {'SINCROG018

In [11]:
print(chronos['CLGA'])


      /-Strpur_NA
   /-|
  |  |   /-Sackow_NA
  |   \-|
  |      \-Ptyfla_NA
  |
  |         /-Parata_chr10
  |        |
  |        |         /-Petmar_chrm31
  |        |      /-|
  |        |     |   \-Parata_chr14
  |      /-|   /-|
  |     |  |  |  |   /-Petmar_chrm53
  |     |  |  |   \-|
  |     |  |  |     |   /-Petmar_chrm33
  |     |  |  |      \-|
  |     |  |  |         \-Parata_chr1
  |     |   \-|
  |     |     |         /-Petmar_chrm6
  |     |     |      /-|
--|     |     |     |   \-Parata_chr9
  |     |     |   /-|
  |     |     |  |  |   /-Petmar_chrm17
  |     |     |  |   \-|
  |     |      \-|      \-Parata_chr4
  |     |        |
  |     |        |   /-Lepocu_alpha2
  |   /-|        |  |
  |  |  |         \-|      /-Xentro_alpha2
  |  |  |           |   /-|
  |  |  |            \-|   \-Galgal_alpha2
  |  |  |              |
  |  |  |               \-Chipun_alpha2
  |  |  |
  |  |  |      /-Lepocu_alpha1
  |  |  |     |
  |  |  |     |      /-Xentro_alpha1
  |  |  